## 0. 今日の量子コンピュータの問題

- Noisy Intermediate-Scale Quantum (NISQ) デバイス
    - 量子回路が深くなる（ゲート数が多くなる）ほど、誤差が大きくなる
    - 十分な量子ビット数ではない
- 量子デバイスは特別なゲート演算のみが用意されている
- 特定のqubits間の量子ビット演算(multi qubit operation)しか用意されていない
- それぞれの量子デバイスに対して、量子ソフトウェアツールキットが用意されてる


### 0-1. TKETとは
- Quantum Software Development Kit
- C++で実装
- pythonモジュール　`pytket`で利用可能
- 最適化コンパイラ：　ユーザーフレンドリーな回路→量子デバイスで実行可能な回路に変換可能
    - Language-agnostic (多くの量子プログラミングフレームワーク(qiskit, Cirq, etc)をサポート)
    - Retagetable (多くの量子デバイス(Quantinuum, IBM, etc)をサポート)
    - Circuit Optimisation (量子計算時に生じるデバイスエラーの影響を最小化。デバイス依存＆デバイス非依存のものが実装)  

#### H2にはpytketで記述した量子回路をNEXUS経由（qnexusを利用）して実行。  
<img src="./fig/tket.png" width="350">

#### HeliosにはGuppy（New Software for quantum computing）で記述した量子回路をNEXUS経由（qnexusを利用）して実行。  
<img src="./fig/guppy.png" width="350">



#### 参照
- [pytket API ドキュメント](https://docs.quantinuum.com/tket/api-docs/)
- [pytket ユーザーガイド](https://docs.quantinuum.com/tket/user-guide/index.html)
- [qnexus API ドキュメント](https://docs.quantinuum.com/nexus/nexus_api/qnexus_api.html)
- [t|ket⟩ : A Retargetable Compiler for NISQ Devices](https://arxiv.org/abs/2003.10611)

### 0-2. このノートブックで必要となる python パッケージ
Python 3.12.11で動作確認をしています。

|  パッケージ （version） |  概要  |
| :---- | :---- |
|  pytket (2.18.1) |  TKETを利用するためのpython モジュール  ( available for python 3.10 or higher )|
|  qnexus (0.48.2) |  Nexusにアクセスし, 量子回路のコンパイルやQuantinuum Hardware/Emulatorへの実行を可能にするpackage  |


In [ ]:
!pip freeze |grep pytket

In [ ]:
!pip freeze |grep qnexus

環境にインストールされていない場合は、以下のセルの＃を取り除き、インストールしてください。

In [ ]:
#!pip install -U pytket #TKET量子回路の作成、量子回路の最適化をじっこうするためのパッケージ
#!pip install -U qnexu #Nexusにアクセスし, 量子回路のコンパイルやQuantinuum Hardware/Emulatorへの実行を可能にするパッケージ
#!pip install -U pylatexenc #可視化のためのパッケージ

## 1. TKET量子回路を作成し、可視化する

In [ ]:
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter

bell = Circuit(2)
bell.H(0).CX(0,1)
bell.measure_all()
render_circuit_jupyter(bell)

## 2. `qnexus`を利用し`TKET`の量子回路を量子デバイス/シミュレータで実行

### 2-1. `qnexus`にログインし、projects/teamsを確認。projectsを指定/作成し、アクティベーションする

In [ ]:
import qnexus as qnx
from pytket import Circuit
from datetime import datetime

In [ ]:
#qnx.login()
qnx.login_with_credentials()

In [ ]:
#projectsを作成/確認
#my_project_ref = qnx.projects.get_or_create("My Project")
all_my_projects = qnx.projects.get_all()
all_my_projects.df()

### 2-2. TKET 量子回路をNexus上のQuantinuumのエミュレータで計算

#### Nexusのprojectsを指定/作成し、projectsをアクティベーション

In [ ]:
my_project_ref = qnx.projects.get_or_create("My Project")
qnx.context.set_active_project(my_project_ref)

#### 作成した量子回路をprojectsに保存

In [ ]:
#Nexusのアクティベーションしたプロジェクトに量子回路を保存
my_circ = qnx.circuits.upload(name="my_circ", circuit=bell)

In [ ]:
#保存した量子回路を表示
my_circ.download_circuit()

In [ ]:
from pytket.circuit.display import render_circuit_jupyter
render_circuit_jupyter(my_circ.download_circuit())

#### 作成した量子回路を量子デバイスで実行するために最適化(コンパイル)

In [ ]:
#ジョブを区別するための時刻を用意
my_job_name_prefix = datetime.now()

In [ ]:
my_comp = qnx.start_compile_job(
    name = f"my_circ compilation {my_job_name_prefix}",
    programs=[my_circ],#複数の量子回路をコンパイルすることが可能 ex. programs=[my_circ1,my_circ2]
    optimisation_level=3,
    backend_config = qnx.QuantinuumConfig(device_name="H2-1LE"), #ノイズなしエミュレータ
#    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator"), #ノイズありエミュレータ
#    project=my_project_ref,
    )

In [ ]:
#my_comp:コンパイル前後の量子回路、どのprojectで実行したなどの情報をもったデータフレーム
my_comp.df()

#### コンパイルジョブの詳細を確認

In [ ]:
qnx.jobs.status(my_comp)
#qnx.jobs.wait_for(my_comp)

In [ ]:
#qnx.jobs.resultsを使ってコンパイルの詳細を確認できる
compile_job_result_refs = qnx.jobs.results(my_comp)
compile_job_result_refs.df()

In [ ]:
# コンパイル前の量子回路（オリジナルの量子回路）
circ_input = compile_job_result_refs[0].get_input().download_circuit()
circ_input

In [ ]:
render_circuit_jupyter(circ_input)

In [ ]:
# コンパイル後の量子回路（H2-1LEで実行できる量子回路）
circ_output = compile_job_result_refs[0].get_output().download_circuit()
circ_output

In [ ]:
render_circuit_jupyter(circ_output)

#### エミュレータ（H2-1LE）に量子回路を実行

In [ ]:
compiled_circuits = [item.get_output() for item in qnx.jobs.results(my_comp)]
my_exe = qnx.start_execute_job(
    programs=compiled_circuits,
    name=f"my_circ execution {my_job_name_prefix}",
    n_shots=[100] * len(compiled_circuits),
    backend_config=qnx.QuantinuumConfig(device_name="H2-1LE"), #ノイズなしエミュレータ
#    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator"), #ノイズありエミュレータ
#    project=my_project_ref,
)

#### 実行ジョブの詳細を確認

In [ ]:
qnx.jobs.status(my_exe)
#qnx.jobs.wait_for(my_exe)

In [ ]:
execute_job_result_refs = qnx.jobs.results(my_exe)

In [ ]:
execute_job_result_refs.df()

In [ ]:
#実行結果
result = execute_job_result_refs[0].download_result()
result.get_counts()

In [ ]:
# 実行ジョブにも実行した量子回路のデータはある
circ_exe = execute_job_result_refs[0].get_input().download_circuit()
circ_exe

In [ ]:
render_circuit_jupyter(circ_exe)

### 2-3. 過去の実行結果を参照

In [ ]:
#pandas dataframe
job_refs = qnx.jobs.get_all().df()

In [ ]:
job_refs[job_refs['created']>'2026-08-01']

In [ ]:
#CompileJobRef
job_comp = qnx.jobs.get(id='XXXX')
job_comp

In [ ]:
compile_job_result_refs1 = qnx.jobs.results(job_comp)
compile_job_result_refs1[0].get_input().download_circuit()

In [ ]:
#ExecuteJobRef
job_exe = qnx.jobs.get(id='YYYY')
job_exe

In [ ]:
execute_job_result_refs1 = qnx.jobs.results(job_exe)
execute_job_result_refs1[0].get_input().download_circuit()

In [ ]:
result = execute_job_result_refs1[0].download_result()
result.get_counts()

## 3. 量子回路の解析、ハードウェアで実行な可能な量子回路なのかをシンタックスチェッカーで確認

In [ ]:
#以下の`CompileJobRef`に対して解析を行う。
job_ref = qnx.jobs.get(id='XXXX')  
job_comp = qnx.jobs.results(job_ref)[0].get_output()  
job_comp

### 3.1 量子回路の解析


In [ ]:
circ = job_comp.download_circuit()
render_circuit_jupyter(circ)

In [ ]:
print("# of qubits:" f'{circ.n_qubits}')
print("# of gates:" f'{circ.n_gates}')
print("# of 1qb gates:" f'{circ.n_1qb_gates()}')
print("# of 2qb gates:" f'{circ.n_2qb_gates()}')
print("circuit depth:" f'{circ.depth()}')
print("circuit 2qb gates depth:" f'{circ.depth_2q()}')


### 3.2 Syntax checkerの利用 H2利用の前には必ず実行ください。  
#### (Jobが問題なく実行可能か、HQCコストなどの確認)

In [ ]:
device_name = "H2-1SC"
config = qnx.QuantinuumConfig(device_name=device_name)
job_name = f"execution-job-qir-{datetime.now()}"
ref_execute_job = qnx.start_execute_job(
    programs=[job_comp],
    n_shots=[1000],
    backend_config=config,
    name=job_name,
)

qnx.jobs.wait_for(ref_execute_job)

In [ ]:
qnx.client.circuits.cost(job_comp,n_shots=1000,backend_config=config)

# 弊社Quantinuumのご紹介
- Website（ 英語 ）： https://www.quantinuum.com/
- ウェブサイト（ 日本語 ）： https://quantinuum.co.jp/
- Press Releases（ 英語 ）： https://www.quantinuum.com/news/news#press-release
- ニュース（ 日本語 ）： https://quantinuum.co.jp/news/
- X（ 日本語 ）： https://x.com/quantinuum_jp
- 採用情報（ 英語 ）：https://www.quantinuum.com/careers


## 補足. 量子回路の変換
pytketでは
- qiskitで記述した量子回路(`qiskit.QuantumCircuit`)からTKETの量子回路のクラスに変換が可能 [qiskit_to_tk](https://docs.quantinuum.com/tket/extensions/pytket-qiskit/#converting-circuits-between-pytket-and-qiskit) 
- TKETで記述した量子回路からqiskitの量子回路(`qiskit.QuantumCircuit`)のクラスに変換が可能 [tk_to_qiskit](https://docs.quantinuum.com/tket/extensions/pytket-qiskit/#converting-circuits-between-pytket-and-qiskit) 

参照：[pytket-qiskit](https://docs.quantinuum.com/tket/extensions/pytket-qiskit/) 